In [ ]:
!pip install mlflow boto3 awscli

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.

In [ ]:
!aws configure

In [ ]:
import mlflow
mlflow.set_tracking_uri("http://3.209.11.118:5000/")

In [ ]:
mlflow.set_experiment('Exp- LinearLVC')

<Experiment: artifact_location='s3://mlflow-bucket-rg/7', creation_time=1771999642152, experiment_id='7', last_update_time=1771999642152, lifecycle_stage='active', name='Exp- LinearLVC', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [ ]:
df = pd.read_csv("/content/reddit_preprocessing.csv").dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [ ]:
df.head()

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [ ]:
vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    max_features=1000,
    min_df=5,
    max_df=0.9
)

In [ ]:
X = vectorizer.fit_transform(df['clean_comment']).toarray()
y = df['category']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Define and train a LinearSVC baseline model using a simple train-test split
with mlflow.start_run() as run:

  # metadata
  mlflow.set_tag("mlflow.runName", "LinearSVC_TrainTestSplit")
  mlflow.set_tag("experiment_type", "baseline 3.0")
  mlflow.set_tag("model_type", "LinearSVC")
  mlflow.set_tag(
      "description",
      "LinearSVC model for sentiment analysis using Bag of Words and bigrams with simple train test split"
  )

  # vectorizer logs (BoW + bigrams only)
  mlflow.log_param("vectorizer_type", "CountVectorizer")
  mlflow.log_param("ngram_range", (1, 2))
  mlflow.log_param("vectorizer_max_features", 10000)
  mlflow.log_param("min_df", 5)
  mlflow.log_param("max_df", 0.9)

  # LinearSVC hyperparams (these are the relevant ones)
  C = 1.0
  class_weight = "balanced"
  max_iter = 5000

  mlflow.log_param("C", C)
  mlflow.log_param("class_weight", class_weight)
  mlflow.log_param("max_iter", max_iter)

  # train
  model = LinearSVC(C=C, class_weight=class_weight, max_iter=max_iter, random_state=42)
  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)

  # metrics
  accuracy = accuracy_score(y_test, y_pred)
  macro_f1 = f1_score(y_test, y_pred, average="macro")

  mlflow.log_metric("accuracy", accuracy)
  mlflow.log_metric("macro_f1", macro_f1)

  report = classification_report(y_test, y_pred, output_dict=True)
  for label, metrics in report.items():
    if isinstance(metrics, dict):
      for metric, value in metrics.items():
        mlflow.log_metric(f"{label}_{metric}", value)

  # confusion matrix artifact
  conf_matrix = confusion_matrix(y_test, y_pred)
  plt.figure(figsize=(8,6))
  sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
  plt.xlabel("Predicted")
  plt.ylabel("Actual")
  plt.title("Confusion Matrix - LinearSVC")
  plt.savefig("/content/confusion_matrix.png")
  plt.close()
  mlflow.log_artifact("/content/confusion_matrix.png")

  # log model artifact
  mlflow.sklearn.log_model(model, "linear_svc_model")

print(f"Accuracy: {accuracy:.4f} | Macro F1: {macro_f1:.4f}")

2026/02/25 06:21:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/25 06:21:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LinearSVC_TrainTestSplit at: http://3.209.11.118:5000/#/experiments/7/runs/d418d30a1d85480dbd4b5b4975a6876b
🧪 View experiment at: http://3.209.11.118:5000/#/experiments/7
Accuracy: 0.7967 | Macro F1: 0.7777


In [ ]:
## Caliberated method

from sklearn.calibration import CalibratedClassifierCV

with mlflow.start_run() as run:

  # metadata
  mlflow.set_tag("mlflow.runName", "LinearSVC_TrainTestSplit")
  mlflow.set_tag("experiment_type", "baseline 3.0")
  mlflow.set_tag("model_type", "LinearSVC")
  mlflow.set_tag(
      "description",
      "LinearSVC model for sentiment analysis using Bag of Words and bigrams with simple train test split"
  )

  # vectorizer logs (BoW + bigrams only)
  mlflow.log_param("vectorizer_type", "CountVectorizer")
  mlflow.log_param("ngram_range", (1, 2))
  mlflow.log_param("vectorizer_max_features", 10000)
  mlflow.log_param("min_df", 5)
  mlflow.log_param("max_df", 0.9)

  # LinearSVC hyperparams (these are the relevant ones)
  # C = 1.0
  class_weight = "balanced"
  max_iter = 5000

  mlflow.log_param("C", C)
  mlflow.log_param("class_weight", class_weight)
  mlflow.log_param("max_iter", max_iter)

  # train
  svc = LinearSVC(C=1.0, class_weight="balanced", max_iter=5000)
  model = CalibratedClassifierCV(svc, method="sigmoid", cv=3)
  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)

  # metrics
  accuracy = accuracy_score(y_test, y_pred)
  macro_f1 = f1_score(y_test, y_pred, average="macro")

  mlflow.log_metric("accuracy", accuracy)
  mlflow.log_metric("macro_f1", macro_f1)

  report = classification_report(y_test, y_pred, output_dict=True)
  for label, metrics in report.items():
    if isinstance(metrics, dict):
      for metric, value in metrics.items():
        mlflow.log_metric(f"{label}_{metric}", value)

  # confusion matrix artifact
  conf_matrix = confusion_matrix(y_test, y_pred)
  plt.figure(figsize=(8,6))
  sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
  plt.xlabel("Predicted")
  plt.ylabel("Actual")
  plt.title("Confusion Matrix - LinearSVC")
  plt.savefig("/content/confusion_matrix.png")
  plt.close()
  mlflow.log_artifact("/content/confusion_matrix.png")

  # log model artifact
  mlflow.sklearn.log_model(model, "linear_svc_model")

print(f"Accuracy: {accuracy:.4f} | Macro F1: {macro_f1:.4f}")

2026/02/25 06:40:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/25 06:40:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LinearSVC_TrainTestSplit at: http://3.209.11.118:5000/#/experiments/7/runs/b5748b86b7e84c72b4190fc49d67c76b
🧪 View experiment at: http://3.209.11.118:5000/#/experiments/7
Accuracy: 0.7969 | Macro F1: 0.7738
